# Unified MIMIC build for COPER + ICU-MDP

This notebook runs the **same pipeline** as the command line:

```bash
python -m data_mngmt
```

It starts from **MIMIC-III CSV files** on disk (PhysioNet layout), drives [YerevaNN/mimic3-benchmarks](https://github.com/YerevaNN/mimic3-benchmarks) (`extract_subjects` → … → `create_in_hospital_mortality` → `split_train_val`), then exports a COPER-style pickle and (optionally) rebuilds ICU-Sepsis MDP parameters.

**Default time framing (this notebook):** **48 hours** of early ICU context and **1 hour** discretization for **IHM tensors** → **COPER training pickle**. The same run prepares the **RL cohort CSV** (`mortality_inhospital` added in Python) and builds **MDP** dynamics from **one transition per table row**. With **`SEPSIS_COHORT = False`** (default in the run cell below), the **IHM listfiles are not** subset to PhysioNet sepsis ICDs — you get the **full mimic3-benchmarks IHM cohort** for COPER, and the prepared MDP CSV keeps **all** rows from the RL snapshot (no extra sepsis filter in `prepare_mdp_cohort_csv`). **`MDP_RL_BLOC_INTERVAL_HOURS` defaults to 1**; passed to `sepsis_cohort.py --bloc-interval-hours`. If `generated/unified/<slug>/mimic_dataset_table_src_bloc1h.csv` already exists, Postgres is **skipped** unless `MDP_FORCE_REBUILD_SOURCE_TABLE=True`. **Scope note:** the **Postgres / mimic_sepsis** pipeline still builds a **sepsis-oriented** `MIMICtable.csv`; it is not “all ICU admissions in MIMIC”. For a **strictly identical** patient set between COPER tensors and MDP, either set **`SEPSIS_COHORT = True`** or supply a custom **`COHORT_CSV`** that matches your COPER `ICUSTAY_ID`s.

## Adjustable parameters (defaults in this notebook: full IHM cohort, 48 h horizon, 60 min = 1 h bins)

| Parameter | Default (this notebook) | Meaning |
| --- | --- | --- |
| **Sepsis cohort** | `False` | If `True`, intersect IHM listfiles + MDP prep with ICU stays whose `HADM_ID` has a sepsis-related ICD-9 in `DIAGNOSES_ICD` (heuristic). If `False`, **full** benchmark IHM listfiles for COPER and **no** sepsis subset filter on the RL CSV rows. |
| **Timestep** | `60` minutes | Passed to IHM export as **1 h** bins (`timestep_minutes / 60`). Must match an available IHM normalizer under `mimic3-benchmarks/mimic3models/in_hospital_mortality/` (stock is `timestep=1.0` h). |
| **Horizon** | `48` hours | IHM / COPER context window (`period_length`). Upstream `create_in_hospital_mortality` is fixed at **48 h**; other values log a warning. |
| **Label** | In-hospital mortality | **COPER**: `y` from mimic3-benchmarks IHM listfiles. **MDP**: terminal outcome uses column `mortality_inhospital` (0/1) merged from `ADMISSIONS.HOSPITAL_EXPIRE_FLAG` via `ICUSTAYS`, so cluster transitions match **IHM**, not `mortality_90d`. |
| **MDP bloc hours** | `1` | Hours per RL row: `mdp_rl_bloc_interval_hours` → `sepsis_cohort --bloc-interval-hours`. Snapshot `mimic_dataset_table_src_bloc<N>h.csv` under the unified slug dir is **reused** on later runs if present (non-empty); delete it or set `mdp_force_rebuild_source_table` to force Postgres. |

## Preset 2 h × 96 h (optional, next code cell)

The **default** notebook flags keep **60 min / 48 h** with **`SEPSIS_COHORT = False`** → slug **`all-60m-h48ihm`**. Set `USE_2H_96H_PRESET = True` for **120 min / 96 h** → slug **`all-120m-h96ihm`** (or `sepsis-…` if you switch `SEPSIS_COHORT` back to `True`), **`MDP_RL_BLOC_INTERVAL_HOURS = 2`**, normalizer **`ihm_ts2.0`**.

**Important — upstream IHM script is fixed at 48 h:** `mimic3benchmark/scripts/create_in_hospital_mortality.py` uses `n_hours=48` to (1) **drop** stays with ICU LOS **&lt; 48 h** and (2) **truncate** exported timeseries to the first **48 h**. If you only change `HORIZON_HOURS` in this notebook without patching that script, the reader may see a **96 h grid with no real events past 48 h** (masks / padding). For a **true** 96 h IHM cohort, edit the vendored copy under `data_mngmt/vendor/mimic3_benchmarks/…` so `process_partition(..., n_hours=96)` matches your horizon, then run with **`REBUILD_FROM_SCRATCH = True`** so `in-hospital-mortality/` is regenerated. With **`n_hours=96`**, stays with **LOS &lt; 96 h** are **excluded** (stricter than 48 h): you **lose** all short stays in that band; compare **listfile row counts** or pickle `details['shapes']` before/after — `notebooks/datasets.ipynb` does not compute that delta automatically yet.

## Step: `extract_subjects` (mimic3-benchmarks) — what it does

When `FORCE_REBUILD_BENCHMARK=True` (or a fresh `generated/unified/<slug>/`), the pipeline runs:

`python -m mimic3benchmark.scripts.extract_subjects <physionet_mimic_root> <benchmark_root>`

with **`benchmark_root` = `data_mngmt/generated/unified/<slug>/root`** (working directory = vendored `mimic3_benchmarks`).

**Behavior (concrete):**

1. Load MIMIC-III CSVs (`PATIENTS`, `ADMISSIONS`, `ICUSTAYS`, `DIAGNOSES_ICD`, `D_ICD_DIAGNOSES`; supports `.csv.gz`).
2. **Filter cohort:** drop ICU stays with ward transfers; keep admissions with exactly **one** ICU stay; merge patient demographics; **age ≥ 18**; attach in-unit and in-hospital mortality flags.
3. Write **aggregate** tables at the root of `benchmark_root`.
4. **Split by `SUBJECT_ID`:** under each numeric folder, write `stays.csv` and phenotype-augmented `diagnoses.csv`.
5. **Stream** default event tables **`CHARTEVENTS`**, **`LABEVENTS`**, **`OUTPUTEVENTS`** (full scans — this dominates runtime) and append rows into each subject’s **`events.csv`** (same schema for all three sources).

**Products on disk immediately after this step** (all under **`data_mngmt/generated/unified/<slug>/root/`**):

| Location | Files | Contents |
| --- | --- | --- |
| **Root of `root/`** | `all_stays.csv` | One row per retained `ICUSTAY_ID` (times, LOS, care unit, age, mortality flags, etc.). |
| | `all_diagnoses.csv` | ICD diagnosis rows restricted to those stays. |
| | `diagnosis_counts.csv` | Per-code counts (from `count_icd_codes`). |
| | `phenotype_labels.csv` | Wide label matrix from HCUP CCS 2015 definitions (`hcup_ccs_2015_definitions.yaml`). |
| **`<SUBJECT_ID>/`** | `stays.csv` | That subject’s ICU stays only. |
| | `diagnoses.csv` | Diagnoses + CCS phenotype columns for that subject. |
| | `events.csv` | Long-format events: `SUBJECT_ID`, `HADM_ID`, `ICUSTAY_ID`, `CHARTTIME`, `ITEMID`, `VALUE`, `VALUEUOM` (built from the three event tables above). |

Downstream steps (`create_in_hospital_mortality`, COPER pickle export, MDP) read from this tree; they are **not** finished when `extract_subjects` alone completes.

## Outputs (full unified run — explicit names)

- **COPER pickle**: `data_mngmt/generated/mortality_coper_<slug>.data` with `<slug>` = `sepsis-60m-h48ihm` or `all-60m-h48ihm`, etc.
- **Working tree**: `data_mngmt/generated/unified/<slug>/` — includes **`root/`** (above), **`in-hospital-mortality/`** (IHM task after `create_in_hospital_mortality` + `split_train_val`), and `unified_build.json`.
- **MDP** (if Postgres succeeds or `COHORT_CSV` points to an existing RL table): snapshot `mimic_dataset_table_src_bloc<N>h.csv` when using DB, then `mdp_cohort_<slug>.csv` and `mdp_params_<slug>/` under that folder.

## Source table contents (ICU-Sepsis schema)

Each row is one **time block** of an ICU stay (default **1 h** spacing in this repo). Key columns: `icustayid`, `bloc`, demographics and vitals/labs (see `icu_sepsis_helpers/mdp_creation/create_rl_table.py` for the exact list: SOFA, SIRS, `input_4hourly`, `output_4hourly`, `max_dose_vaso`, etc.). The unified build **adds** `mortality_inhospital` from PhysioNet CSVs before `build_mdp`.

## COPER ↔ MDP contract (granularity / variables)

- **Documentation:** `data_mngmt/DATASET_PIPELINE.md` (RL table, default 1 h blocs aligned with COPER timestep, SOFA on the MDP side, join on `ICUSTAY_ID`).
- **JSON:** after a run, `data_mngmt/generated/unified/<slug>/unified_build.json` includes `pipeline_contract` + `rl_table_contents_summary`.

## Prerequisites

- **PhysioNet CSVs:** set `paths.json` → `physionet_mimic_root` to the folder that **directly contains** `PATIENTS.csv` (or `PATIENTS.csv.gz`), `ADMISSIONS.csv` / `.gz`, … (standard PhysioNet layout: `.../physionet.org/files/mimiciii/1.4/`). **Not** the parent `physionet.org` directory—`mimic3-benchmarks` passes this path to `extract_subjects` as the MIMIC CSV root.
- **mimic3-benchmarks:** one-time `python -m data_mngmt.mimic.mimic3_benchmarks_vendor` then `pip install -r data_mngmt/vendor/mimic3_benchmarks/requirements.txt` (vendored under `data_mngmt/vendor/mimic3_benchmarks/`, gitignored). Optional override: env `MIMIC3_BENCHMARKS_REPO`.
- **Training pickle path:** after a successful run, point `paths.json` → `mimic3_mortality` at `data_mngmt/generated/mortality_coper_<slug>.data` (this notebook’s default slug → **`all-60m-h48ihm`**).
- **MDP / RL table:** with `MDP_REBUILD_TABLE_FROM_DB=True` (default), **microsoft/mimic_sepsis** on **Postgres** runs only when no valid `mimic_dataset_table_src_bloc<N>h.csv` snapshot exists (or `MDP_FORCE_REBUILD_SOURCE_TABLE=True`). **`MDP_SKIP_PREPROCESS`:** use **`False`** for the **first** MDP build under a new `unified/<slug>/` (empty `mimic_sepsis_build/processed_files/`). Set **`True`** only when that folder already has `*.csv` from a prior run. Alternative: `MDP_REBUILD_TABLE_FROM_DB=False` and `COHORT_CSV` pointing to an existing RL CSV. Vendor if needed: `python -m data_mngmt.sepsis_rl.mimic_sepsis_vendor`.
- First PhysioNet→benchmark run is often **many hours** on a full MIMIC copy (mostly `extract_subjects` over the three event tables). Set `REBUILD_FROM_SCRATCH=False` and `FORCE_REBUILD_BENCHMARK=False` to reuse `generated/unified/<slug>/` when the benchmark `root/` tree is already complete.


In [1]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

def find_coper_repo(start: Path | None = None) -> Path:
    cwd = (start or Path.cwd()).resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data_mngmt" / "pipeline" / "unified_build.py").is_file() and (p / "paths.json").is_file():
            return p
    raise RuntimeError("Could not find COPER repo (need data_mngmt/pipeline/unified_build.py + paths.json).")


REPO = find_coper_repo()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
print("REPO", REPO)


REPO /home/charlesv/Desktop/StatisitcalGenetics/code/COPER


In [2]:
# Optional: PostgreSQL credentials for MDP rebuild (mimic_sepsis)
# Loaded from paths.json -> postgres_login_json (default: ../../MIMIC/physionet/login.json).
# Template for reproducibility: code/COPER/postgres_login.dummy.json
import os
from data_mngmt import load_postgres_login, postgres_login_path

pg = load_postgres_login(strict=False)
if pg:
    for k, v in pg.items():
        os.environ[k] = v
    print(f"Loaded Postgres login from: {postgres_login_path()}")
else:
    print("No Postgres login loaded. Fill MIMIC/physionet/login.json (or update paths.json -> postgres_login_json).")

for k in ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]:
    print(k, "OK" if os.environ.get(k) else "MISSING")

Loaded Postgres login from: /home/charlesv/Desktop/StatisitcalGenetics/MIMIC/physionet/login.json
PGHOST OK
PGPORT OK
PGDATABASE OK
PGUSER OK
PGPASSWORD OK


In [ ]:
from pathlib import Path

from data_mngmt.pipeline.unified_build import UnifiedBuildParams, build_slug, run_unified_build

# --- Typical workflows ---
# * Iterative (fast): REBUILD_FROM_SCRATCH=False, MDP_FORCE_REBUILD_SOURCE_TABLE=False
#   → reuses benchmark_root + mimic_dataset_table_src_bloc<N>h.csv when present.
# * Full benchmark from CSVs: REBUILD_FROM_SCRATCH=True (many hours, CHARTEVENTS/LAB/OUTPUT scans).
# * First MDP for a NEW slug (e.g. all-60m-h48ihm): MDP_SKIP_PREPROCESS=False so Postgres preprocess.py runs
#   (each slug has its own generated/unified/<slug>/mimic_sepsis_build/processed_files/).

REBUILD_FROM_SCRATCH = False  # if True, rebuild benchmark + IHM from PhysioNet CSVs (very long)

# Full benchmark IHM cohort for COPER (no PhysioNet sepsis ICD filter on listfiles).
# Set True to restrict to sepsis-flagged HADM_IDs → slug prefix ``sepsis-`` instead of ``all-``.
SEPSIS_COHORT = False

# 1 h × 48 h IHM (stock listfiles). True → 2 h × 96 h (slug e.g. all-120m-h96ihm; patch upstream n_hours for real 96 h data).
USE_2H_96H_PRESET = False

if USE_2H_96H_PRESET:
    TIMESTEP_MINUTES = 120
    HORIZON_HOURS = 96
    MDP_RL_BLOC_INTERVAL_HOURS = 2.0
else:
    TIMESTEP_MINUTES = 60
    HORIZON_HOURS = 48
    MDP_RL_BLOC_INTERVAL_HOURS = 1.0

BUILD_MDP = True
FORCE_REBUILD_BENCHMARK = REBUILD_FROM_SCRATCH

COHORT_CSV = None  # Path to RL table CSV, or None to use DB / cached snapshot
MDP_REBUILD_TABLE_FROM_DB = True  # False requires a valid COHORT_CSV for MDP
# False = run preprocess.py against Postgres when needed. True only if this slug already has
#   generated/unified/<slug>/mimic_sepsis_build/processed_files/*.csv from a previous run.
MDP_SKIP_PREPROCESS = False
MDP_FORCE_REBUILD_SOURCE_TABLE = False  # True = ignore snapshot; always run mimic_sepsis (long)

_slug_probe = build_slug(
    UnifiedBuildParams(
        sepsis_cohort=SEPSIS_COHORT,
        timestep_minutes=TIMESTEP_MINUTES,
        horizon_hours=HORIZON_HOURS,
    )
)
_mdp_pf = REPO / "data_mngmt" / "generated" / "unified" / _slug_probe / "mimic_sepsis_build" / "processed_files"
_need_mdp_db = BUILD_MDP and MDP_REBUILD_TABLE_FROM_DB and COHORT_CSV is None
if MDP_SKIP_PREPROCESS and _need_mdp_db:
    if not _mdp_pf.is_dir() or not any(_mdp_pf.glob("*.csv")):
        print(
            "MDP_SKIP_PREPROCESS was True but no Postgres extracts for this slug; running preprocess.",
            _mdp_pf,
        )
        MDP_SKIP_PREPROCESS = False

params = UnifiedBuildParams(
    sepsis_cohort=SEPSIS_COHORT,
    timestep_minutes=TIMESTEP_MINUTES,
    horizon_hours=HORIZON_HOURS,
    build_mdp=BUILD_MDP,
    force_rebuild_benchmark=FORCE_REBUILD_BENCHMARK,
    cohort_csv=Path(COHORT_CSV) if COHORT_CSV else None,
    mdp_rebuild_table_from_db=MDP_REBUILD_TABLE_FROM_DB,
    mdp_skip_preprocess=MDP_SKIP_PREPROCESS,
    mdp_force_rebuild_source_table=MDP_FORCE_REBUILD_SOURCE_TABLE,
    mdp_rl_bloc_interval_hours=MDP_RL_BLOC_INTERVAL_HOURS,
)

_slug = build_slug(params)
_pickle = REPO / "data_mngmt" / "generated" / f"mortality_coper_{_slug}.data"
print("Unified build slug:", _slug)
print("COPER pickle (set paths.json -> mimic3_mortality):", _pickle)

result = run_unified_build(params)
print(json.dumps(result, indent=2, default=str))

INFO Reusing existing benchmark_root: /home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/generated/unified/all-60m-h48ihm/root
INFO Reusing existing IHM task dir: /home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/generated/unified/all-60m-h48ihm/in-hospital-mortality


Unified build slug: all-60m-h48ihm
COPER pickle (set paths.json -> mimic3_mortality): /home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/generated/mortality_coper_all-60m-h48ihm.data


/home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/vendor/mimic3_benchmarks/mimic3models/preprocessing.py:216: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dct = pickle.load(load_file, encoding='latin1')
INFO Mapped 42019 stay stems -> ICUSTAY_ID (skipped empty=0 invalid=6)
INFO Running preprocess (cwd=/home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/generated/unified/all-60m-h48ihm/mimic_sepsis_build) …
INFO Running mimic_sepsis script with pseudo-TTY (progress output forwarded to stderr).
/home/charlesv/Desktop/StatisitcalGenetics/code/COPER/data_mngmt/vendor/mimic_sepsis_upstream/preprocess.py:142: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  d = pd.read_sql_q